# Ciencia de Datos — RDA 2
## Extracción de datos con Python

**Duración estimada:** 3 horas &nbsp;|&nbsp; **Entorno:** Jupyter Notebook &nbsp;|&nbsp; **Bibliotecas:** Pandas y Requests

---

### Objetivo de aprendizaje

Aplicar técnicas de extracción de datos desde archivos estructurados y servicios externos, utilizando Python para obtener, cargar y verificar conjuntos de datos destinados a procesos posteriores de análisis.

### Herramientas

| Herramienta | Uso en esta práctica |
|---|---|
| Python | Lenguaje base de todos los ejemplos |
| Pandas | Lectura de CSV, Excel y JSON; construcción de DataFrames |
| Requests | Envío de solicitudes HTTP a servicios externos (API REST) |
| JSON | Módulo estándar para interpretar estructuras semiestructuradas |
| Jupyter Notebook | Entorno interactivo de trabajo y documentación |

### Escenario de trabajo

Durante toda la sesión se trabaja con un mismo dominio: **productos e inventario de una empresa de tecnología**. De esta forma la atención se centra en los mecanismos de extracción y no en comprender un dominio diferente en cada ejemplo.

### Alcance de la sesión

Esta clase se concentra en la etapa de **extracción**. Después de obtener los datos se realizan inspecciones y verificaciones básicas, pero **no** se desarrolla todavía un proceso de limpieza o transformación; eso corresponde a sesiones posteriores.

### Distribución de la sesión

| Sección | Tiempo estimado |
|---|---|
| 0. Preparación del entorno | 5 min |
| 1. Fundamentos de extracción | 20 min |
| 2. Extracción CSV | 25 min |
| 3. Extracción Excel | 15 min |
| 4. Extracción JSON | 25 min |
| 5. Extracción mediante API REST | 45 min |
| 6. Comparación de fuentes | 10 min |
| 7. Validación de datos | 15 min |
| 8. Práctica integradora | 20 min |
| 9. Conclusiones | 5 min |
| **Total** | **180 min** |

---
## 0. Preparación del entorno

**Objetivo:** disponer de las bibliotecas necesarias para ejecutar los ejemplos.

Contenido de esta sección:

- Instalación de dependencias cuando sea necesario.
- Importación de bibliotecas.
- Organización de directorios y definición de rutas de archivos.
- Generación de los archivos de práctica.

### 0.1. Instalación de dependencias

Si alguna biblioteca no está instalada, quite el símbolo `#` de la siguiente celda y ejecútela una sola vez. La biblioteca `openpyxl` es necesaria para que Pandas pueda leer y escribir archivos Excel (`.xlsx`).

In [ ]:
# %pip install pandas requests openpyxl

### 0.2. Importación de bibliotecas

In [ ]:
import pandas as pd
import json
import requests
from pathlib import Path

print("Pandas  :", pd.__version__)
print("Requests:", requests.__version__)

### 0.3. Organización de directorios y rutas

Se recomienda la siguiente estructura de carpetas, donde el notebook y los datos están separados:

```
proyecto/
├── notebooks/
│   └── extraccion_datos.ipynb
└── datasets/
    ├── productos.csv
    ├── inventario_reporte.csv
    ├── ventas.xlsx
    ├── productos.json
    └── catalogo.json
```

La ruta `../datasets` significa: "subir un nivel desde la carpeta del notebook y entrar en `datasets`". El objeto `Path` permite construir rutas que funcionan igual en Windows, Linux y macOS.

In [ ]:
DATA_DIR = Path("../datasets")
DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Directorio de datos:", DATA_DIR.resolve())

### 0.4. Generación de los archivos de práctica

En un escenario real, estos archivos serían entregados por la empresa o descargados de sus sistemas. Para que el notebook sea reproducible, la siguiente celda los genera con datos simulados del escenario de inventario. Los archivos se crean **una sola vez**: si ya existen, no se sobrescriben.

Los datos incluyen, de forma intencional, algunos valores faltantes, registros duplicados y caracteres con tilde, porque son situaciones habituales en fuentes reales.

> Esta celda no forma parte del contenido a evaluar; basta con ejecutarla.

In [ ]:
import random
from datetime import date, timedelta


def generar_archivos_practica(directorio: Path, sobrescribir: bool = False) -> None:
    """Genera los archivos simulados del escenario de inventario."""
    archivos = ["productos.csv", "inventario_reporte.csv", "ventas.xlsx",
                "productos.json", "catalogo.json"]
    if not sobrescribir and all((directorio / a).exists() for a in archivos):
        print("Los archivos de práctica ya existen. No se generaron nuevamente.")
        return

    random.seed(42)

    categorias = {1: "Computación", 2: "Periféricos", 3: "Almacenamiento",
                  4: "Redes", 5: "Audio y video"}
    tipos = {
        "Laptop": (1, 850), "Tablet": (1, 320), "Monitor": (2, 210),
        "Teclado": (2, 35), "Mouse": (2, 18), "Impresora": (2, 160),
        "Disco SSD": (3, 75), "Memoria USB": (3, 12), "Router": (4, 60),
        "Audífonos": (5, 45), "Cámara web": (5, 55), "Parlante": (5, 40),
    }
    marcas = ["Lenovo", "HP", "Dell", "Logitech", "Epson", "TP-Link",
              "Kingston", "Samsung", "Sony", "Xiaomi"]
    proveedores = ["Distribuidora Andina", "TecnoImport S.A.",
                   "Comercial Sierra", "Mayorista Pacífico"]
    etiquetas_por_categoria = {
        1: ["computacion", "portatil", "oficina"],
        2: ["perifericos", "oficina", "accesorios"],
        3: ["almacenamiento", "respaldo"],
        4: ["redes", "conectividad"],
        5: ["multimedia", "audio", "video"],
    }

    # ---------- productos.csv ----------
    filas = []
    inicio = date(2025, 1, 1)
    for i in range(1, 121):
        tipo = random.choice(list(tipos))
        cat_id, precio_base = tipos[tipo]
        marca = random.choice(marcas)
        filas.append({
            "id_producto": i,
            "producto": f"{tipo} {marca} {random.randint(100, 999)}",
            "categoria_id": cat_id,
            "precio": round(precio_base * random.uniform(0.8, 1.4), 2),
            "stock": random.randint(0, 150),
            "proveedor": random.choice(proveedores),
            "fecha_ingreso": (inicio + timedelta(days=random.randint(0, 600))).isoformat(),
        })
    df = pd.DataFrame(filas)
    df.loc[[7, 33, 81], "precio"] = None        # precios faltantes
    df.loc[[15, 64], "proveedor"] = None        # proveedores faltantes
    df = pd.concat([df, df.iloc[[10, 42]]], ignore_index=True)  # duplicados
    df.to_csv(directorio / "productos.csv", index=False, encoding="utf-8")

    # ---------- inventario_reporte.csv (formato "de sistema heredado") ----------
    lineas = ["Reporte de inventario - Bodega central",
              "Generado: 2026-11-01",
              "codigo;descripcion;existencias;costo"]
    for _, fila in df.head(25).iterrows():
        costo = "" if pd.isna(fila["precio"]) else f"{fila['precio'] * 0.7:.2f}".replace(".", ",")
        lineas.append(f"P{fila['id_producto']:04d};{fila['producto']};{fila['stock']};{costo}")
    (directorio / "inventario_reporte.csv").write_text("\n".join(lineas) + "\n", encoding="utf-8")

    # ---------- ventas.xlsx (tres hojas) ----------
    ventas = []
    vendedores = ["Andrea Ruiz", "Carlos Vega", "María Torres", "Luis Paredes"]
    base = df.drop_duplicates().dropna(subset=["precio"]).reset_index(drop=True)
    for _ in range(200):
        p = base.sample(1, random_state=random.randint(0, 10_000)).iloc[0]
        cantidad = random.randint(1, 5)
        ventas.append({
            "fecha": inicio + timedelta(days=random.randint(0, 650)),
            "producto": p["producto"],
            "cantidad": cantidad,
            "precio_unitario": p["precio"],
            "total": round(cantidad * p["precio"], 2),
            "vendedor": random.choice(vendedores),
        })
    df_ventas = pd.DataFrame(ventas).sort_values("fecha").reset_index(drop=True)
    df_ventas.loc[[5, 90, 150], "vendedor"] = None   # celdas vacías

    df_vendedores = pd.DataFrame({
        "vendedor": vendedores,
        "sucursal": ["Ambato", "Latacunga", "Ambato", "Píllaro"],
        "meta_mensual": [5000, 4500, 5200, 3800],
    })

    with pd.ExcelWriter(directorio / "ventas.xlsx", engine="openpyxl") as writer:
        df_ventas.to_excel(writer, sheet_name="Ventas", index=False)
        df_vendedores.to_excel(writer, sheet_name="Vendedores", index=False)
        # Hoja con un título antes de la tabla (situación frecuente en reportes)
        pd.DataFrame([["Resumen anual de ventas"], ["Fuente: sistema comercial"]]).to_excel(
            writer, sheet_name="Resumen", index=False, header=False)
        resumen = df_ventas.assign(anio=pd.to_datetime(df_ventas["fecha"]).dt.year)
        resumen = resumen.groupby("anio", as_index=False)["total"].sum().round(2)
        resumen.to_excel(writer, sheet_name="Resumen", index=False, startrow=3)

    # ---------- productos.json (lista de objetos con anidamiento) ----------
    productos_json = []
    for _, fila in df.drop_duplicates().head(30).iterrows():
        cat_id = int(fila["categoria_id"])
        caracteristicas = {
            "marca": fila["producto"].split()[-2],
            "garantia_meses": random.choice([6, 12, 24]),
            "dimensiones": {"ancho_cm": round(random.uniform(5, 60), 1),
                            "alto_cm": round(random.uniform(1, 45), 1)},
        }
        if random.random() > 0.25:          # algunos productos no informan el peso
            caracteristicas["peso_kg"] = round(random.uniform(0.05, 6), 2)
        productos_json.append({
            "id_producto": int(fila["id_producto"]),
            "producto": fila["producto"],
            "disponible": bool(fila["stock"] > 0),
            "categoria": {"id": cat_id, "nombre": categorias[cat_id]},
            "caracteristicas": caracteristicas,
            "etiquetas": random.sample(etiquetas_por_categoria[cat_id], k=2),
        })
    with open(directorio / "productos.json", "w", encoding="utf-8") as f:
        json.dump(productos_json, f, ensure_ascii=False, indent=2)

    # ---------- catalogo.json (objeto "envoltorio" con metadatos) ----------
    catalogo = {
        "empresa": "TecnoAndes Cía. Ltda.",
        "fecha_generacion": "2026-11-01",
        "total_productos": 5,
        "productos": [
            {"id_producto": p["id_producto"], "producto": p["producto"],
             "categoria": p["categoria"]["nombre"],
             "proveedores": [{"nombre": n, "pais": "Ecuador"}
                             for n in random.sample(proveedores, k=2)]}
            for p in productos_json[:5]
        ],
    }
    with open(directorio / "catalogo.json", "w", encoding="utf-8") as f:
        json.dump(catalogo, f, ensure_ascii=False, indent=2)

    print("Archivos generados en:", directorio.resolve())


generar_archivos_practica(DATA_DIR)
sorted(p.name for p in DATA_DIR.iterdir())

---
## 1. Fundamentos de extracción de datos

**Objetivo:** comprender qué significa extraer datos y cuál es su función dentro del ciclo de vida de los datos.

### 1.1. Conceptos fundamentales

**¿Qué es una fuente de datos?**
Es el lugar o sistema donde los datos se originan o se almacenan: un archivo, una base de datos, un sensor, una hoja de cálculo o un servicio web. Ejemplo: el sistema de facturación de una empresa.

**¿Qué es la extracción de datos?**
Es el proceso de obtener datos desde una o varias fuentes y llevarlos a un entorno de trabajo (en este caso, Python) en una forma que permita procesarlos. Extraer no significa analizar: significa **obtener y cargar** correctamente.

**Diferencia entre fuente de datos y conjunto de datos**

| Fuente de datos | Conjunto de datos (dataset) |
|---|---|
| Origen de la información | Colección concreta de datos obtenida de una fuente |
| Puede cambiar con el tiempo | Representa los datos en un momento dado |
| Ejemplo: la API de un proveedor | Ejemplo: el DataFrame con los productos consultados hoy |

**Datos internos y externos**

- **Internos:** generados por la propia organización (ventas, inventario, nómina).
- **Externos:** provienen de terceros (proveedores, servicios públicos, APIs, portales de datos abiertos).

**Datos estructurados, semiestructurados y no estructurados**

| Tipo | Característica | Ejemplos |
|---|---|---|
| Estructurados | Filas y columnas con esquema fijo | CSV, Excel, tablas SQL |
| Semiestructurados | Tienen etiquetas o claves, pero la estructura es flexible y puede anidarse | JSON, XML |
| No estructurados | No siguen un modelo de datos predefinido | Texto libre, imágenes, audio |

### 1.2. Extracción dentro del proceso ETL

ETL es el proceso mediante el cual los datos se trasladan desde sus fuentes hasta un destino donde pueden ser analizados.

| | Etapa | Propósito |
|:---:|---|---|
| **E** | **Extract** | Obtener datos desde las fuentes |
| **T** | **Transform** | Modificar datos: limpiar, convertir tipos, combinar, calcular |
| **L** | **Load** | Almacenar datos en el destino: base de datos, data warehouse, archivo |

Representación conceptual del proceso ETL:

```
  FUENTES                 E                 T                   L               DESTINO
 ┌──────────┐      ┌────────────┐     ┌─────────────┐     ┌──────────┐     ┌──────────────┐
 │ CSV      │      │            │     │             │     │          │     │ Base de datos│
 │ Excel    │ ───> │  Extract   │ ──> │  Transform  │ ──> │   Load   │ ──> │ Data         │
 │ JSON     │      │ Obtener    │     │ Modificar   │     │ Almacenar│     │ warehouse    │
 │ API REST │      │            │     │             │     │          │     │ Archivo      │
 └──────────┘      └────────────┘     └─────────────┘     └──────────┘     └──────────────┘
                     Foco de esta sesión
```

> **Aclaración:** este notebook se concentra principalmente en la **primera etapa (Extract)**. Después de obtener los datos se realizan verificaciones básicas para confirmar que la extracción produjo un resultado utilizable, pero la transformación y la carga se abordarán en sesiones posteriores.

### 1.3. Fuentes y mecanismos de extracción

| Fuente | Mecanismo | Formato | Función de Python |
|---|---|---|---|
| Archivos locales | Lectura directa | CSV, Excel | `pd.read_csv()`, `pd.read_excel()` |
| Archivos semiestructurados | Interpretación de estructura | JSON | `json.load()`, `pd.read_json()`, `pd.json_normalize()` |
| Servicios externos | Solicitudes HTTP | JSON | `requests.get()` |
| Bases de datos | Consultas SQL | Tablas | `pd.read_sql()` |

En esta sesión se trabajan las tres primeras fuentes. La extracción desde bases de datos mediante SQL se menciona como referencia y se abordará por separado.

---
## 2. Extracción desde archivos CSV

**Objetivo:** obtener información almacenada en archivos delimitados.

### 2.1. Características del formato CSV

CSV (*Comma-Separated Values*) es un archivo de **texto plano** donde cada línea es un registro y los valores se separan con un carácter delimitador.

- **Organización mediante filas y columnas:** cada línea es una fila; cada valor separado es una columna.
- **Separadores:** el más común es la coma (`,`), pero también se usan punto y coma (`;`), tabulación (`\t`) o barra vertical (`|`). En países que usan la coma como separador decimal es frecuente encontrar `;`.
- **Encabezados:** la primera línea suele contener los nombres de las columnas, aunque no es obligatorio.
- **Codificación de caracteres:** define cómo se representan letras con tilde, la ñ y otros símbolos. Las más comunes son `utf-8` y `latin-1` (ISO-8859-1).
- **Representación de valores nulos:** un valor faltante puede aparecer como campo vacío, `NA`, `NULL`, `-`, etc.

Antes de leer un archivo con Pandas, conviene mirar sus primeras líneas como texto:

In [ ]:
with open(DATA_DIR / "productos.csv", "r", encoding="utf-8") as archivo:
    for _ in range(5):
        print(archivo.readline(), end="")

### 2.2. Lectura de archivos

La función `pd.read_csv()` lee el archivo y devuelve un **DataFrame**. El método `head()` muestra las primeras cinco filas.

In [ ]:
df_csv = pd.read_csv(
    DATA_DIR / "productos.csv"
)
df_csv.head()

In [ ]:
print("Filas y columnas:", df_csv.shape)

### 2.3. Parámetros de extracción

| Parámetro | Función |
|---|---|
| `sep` | Define el separador |
| `encoding` | Define la codificación |
| `header` | Identifica la fila de encabezados |
| `names` | Establece nombres de columnas |
| `usecols` | Selecciona columnas |
| `nrows` | Limita la cantidad de filas |
| `skiprows` | Omite filas |

Ejemplo combinando varios parámetros: se extraen solamente tres columnas y las primeras 100 filas.

In [ ]:
df_csv = pd.read_csv(
    DATA_DIR / "productos.csv",
    sep=",",
    encoding="utf-8",
    usecols=["producto", "precio", "stock"],
    nrows=100
)
print("Filas y columnas:", df_csv.shape)
df_csv.head()

#### Caso práctico: un archivo con formato distinto

El archivo `inventario_reporte.csv` fue exportado desde un sistema antiguo. Primero se observa su contenido como texto:

In [ ]:
print((DATA_DIR / "inventario_reporte.csv").read_text(encoding="utf-8")[:300])

Se identifican tres diferencias respecto a `productos.csv`:

1. Las dos primeras líneas son **metadatos**, no datos.
2. El separador es **punto y coma** (`;`).
3. El costo usa **coma decimal** (`12,50`).

Si se intenta leer con los valores por defecto, la extracción falla o produce una estructura incorrecta:

In [ ]:
try:
    df_reporte = pd.read_csv(DATA_DIR / "inventario_reporte.csv")
    print("Forma obtenida:", df_reporte.shape)
    display(df_reporte.head())
except Exception as error:
    print("Error durante la lectura:", type(error).__name__)
    print(error)

Con los parámetros adecuados (`sep`, `skiprows` y `decimal`) la extracción es correcta:

In [ ]:
df_reporte = pd.read_csv(
    DATA_DIR / "inventario_reporte.csv",
    sep=";",
    skiprows=2,      # omite las dos líneas de metadatos
    decimal=","      # interpreta la coma como separador decimal
)
print(df_reporte.dtypes)
df_reporte.head()

Los parámetros `header` y `names` permiten **reemplazar** los nombres de columnas originales por otros más descriptivos. Con `header=0` se indica que la primera fila leída contiene encabezados, y `names` los sustituye:

In [ ]:
df_reporte = pd.read_csv(
    DATA_DIR / "inventario_reporte.csv",
    sep=";",
    skiprows=2,
    decimal=",",
    header=0,
    names=["codigo_producto", "descripcion", "unidades", "costo_unitario"]
)
df_reporte.head()

Cuando un archivo **no tiene encabezados**, se usa `header=None`. Pandas asigna números como nombres de columnas. En el siguiente ejemplo se omiten tres líneas (incluida la de encabezados) para simular ese caso:

In [ ]:
pd.read_csv(
    DATA_DIR / "inventario_reporte.csv",
    sep=";",
    skiprows=3,
    header=None,
    nrows=5
)

### 2.4. Interpretación

Las siguientes celdas permiten observar el efecto de algunos errores comunes antes de responder las preguntas.

**Separador incorrecto:** se lee `productos.csv` indicando `;` cuando el separador real es `,`.

In [ ]:
df_error_sep = pd.read_csv(DATA_DIR / "productos.csv", sep=";")
print("Forma obtenida:", df_error_sep.shape)
df_error_sep.head(3)

**Codificación incorrecta:** se lee un archivo guardado en `utf-8` como si estuviera en `latin-1`. Observe las palabras con tilde.

In [ ]:
df_error_enc = pd.read_csv(DATA_DIR / "productos.csv", encoding="latin-1")
df_error_enc[df_error_enc["producto"].str.contains("Ã", na=False)].head()

**Extracción limitada vs. completa:** comparación de memoria utilizada.

In [ ]:
df_completo = pd.read_csv(DATA_DIR / "productos.csv")
df_parcial = pd.read_csv(DATA_DIR / "productos.csv", usecols=["producto", "precio"], nrows=50)

print(f"Completo: {df_completo.shape} -> {df_completo.memory_usage(deep=True).sum() / 1024:.1f} KB")
print(f"Parcial : {df_parcial.shape} -> {df_parcial.memory_usage(deep=True).sum() / 1024:.1f} KB")

**Preguntas:**

1. ¿Qué sucede cuando se utiliza un separador incorrecto?
2. ¿Por qué puede aparecer texto ilegible?
3. ¿En qué situaciones conviene extraer solamente determinadas columnas?
4. ¿Cuál es la diferencia entre extraer todos los registros y limitar la extracción?

**Respuesta:**

_Escriba aquí su respuesta._

---
## 3. Extracción desde archivos Excel

**Objetivo:** trabajar con fuentes de datos organizadas en hojas de cálculo.

### 3.1. Características de Excel

- **Hojas de trabajo:** un mismo archivo puede contener varias hojas, cada una con información distinta.
- **Encabezados:** no siempre están en la primera fila; a veces hay títulos o notas encima de la tabla.
- **Celdas vacías:** se interpretan como valores nulos (`NaN`) al cargarlas en Pandas.
- **Múltiples tablas en un mismo archivo:** una hoja puede contener más de una tabla o una tabla con filas de resumen, lo que obliga a indicar exactamente qué rango leer.

A diferencia del CSV, un archivo `.xlsx` **no es texto plano**: es un formato binario comprimido que conserva tipos de datos (fechas, números), formatos y fórmulas. Por eso Pandas necesita la biblioteca `openpyxl` para leerlo.

### 3.2. Lectura de archivos Excel

Sin parámetros adicionales, `pd.read_excel()` lee **solo la primera hoja**.

In [ ]:
df_excel = pd.read_excel(
    DATA_DIR / "ventas.xlsx"
)
df_excel.head()

Para conocer qué hojas contiene el archivo antes de extraer:

In [ ]:
archivo_excel = pd.ExcelFile(DATA_DIR / "ventas.xlsx")
archivo_excel.sheet_names

Con `sheet_name=None` se extraen **todas** las hojas a la vez. El resultado es un diccionario cuya clave es el nombre de la hoja y cuyo valor es un DataFrame:

In [ ]:
hojas = pd.read_excel(DATA_DIR / "ventas.xlsx", sheet_name=None)

for nombre, df_hoja in hojas.items():
    print(f"{nombre:<12} -> {df_hoja.shape}")

### 3.3. Extracción selectiva

Se indica la hoja y las columnas que se necesitan:

In [ ]:
df_excel = pd.read_excel(
    DATA_DIR / "ventas.xlsx",
    sheet_name="Ventas",
    usecols=["fecha", "producto", "total"]
)
print(df_excel.dtypes)
df_excel.head()

Observe que la columna `fecha` llega directamente con tipo `datetime64`: Excel conserva el tipo de dato, mientras que en un CSV la fecha llegaría como texto.

**Hoja con título sobre la tabla.** La hoja `Resumen` tiene dos líneas de texto antes de la tabla. Leída sin parámetros, la estructura es incorrecta:

In [ ]:
pd.read_excel(DATA_DIR / "ventas.xlsx", sheet_name="Resumen")

Con `header=3` se indica que los encabezados reales están en la cuarta fila (el conteo empieza en 0):

In [ ]:
pd.read_excel(DATA_DIR / "ventas.xlsx", sheet_name="Resumen", header=3)

**Celdas vacías:** la columna `vendedor` de la hoja `Ventas` tiene celdas sin valor.

In [ ]:
df_ventas = pd.read_excel(DATA_DIR / "ventas.xlsx", sheet_name="Ventas")
df_ventas.isnull().sum()

### 3.4. Comparación entre CSV y Excel

| Aspecto | CSV | Excel (.xlsx) |
|---|---|---|
| Tipo de archivo | Texto plano | Binario comprimido |
| Hojas | Una sola tabla | Varias hojas por archivo |
| Tipos de datos | No los conserva; todo es texto | Conserva fechas, números y formatos |
| Tamaño y velocidad | Liviano y rápido de leer | Más pesado y lento |
| Dependencias en Python | Ninguna adicional | Requiere `openpyxl` |
| Compatibilidad | Universal (cualquier sistema o lenguaje) | Depende de software o bibliotecas compatibles |
| Riesgos frecuentes | Separador y codificación incorrectos | Encabezados desplazados, celdas combinadas, varias tablas en una hoja |
| Escenario de uso | Intercambio entre sistemas, exportaciones masivas | Reportes administrativos, información preparada por personas |

**Pregunta:** si un sistema debe intercambiar diariamente 2 millones de registros con otro sistema, ¿qué formato recomendaría y por qué? ¿Y para un reporte mensual que revisará el gerente de ventas?

**Respuesta:**

_Escriba aquí su respuesta._

---
## 4. Extracción desde archivos JSON

**Objetivo:** interpretar información semiestructurada y comprender cómo se representa mediante pares clave-valor y estructuras anidadas.

### 4.1. Fundamentos de JSON

JSON (*JavaScript Object Notation*) es un formato de texto para representar datos mediante pares **clave-valor**. Es el formato más usado en el intercambio de datos entre aplicaciones web.

| Elemento | Sintaxis | Equivalente en Python |
|---|---|---|
| Objeto | `{ "clave": valor }` | `dict` |
| Arreglo | `[ valor1, valor2 ]` | `list` |
| Clave | Siempre texto entre comillas dobles | `str` |
| Cadena | `"Laptop"` | `str` |
| Número | `850`, `12.5` | `int`, `float` |
| Booleano | `true`, `false` | `True`, `False` |
| Nulo | `null` | `None` |

Un **objeto anidado** es un objeto dentro de otro objeto. En el siguiente ejemplo, `categoria` es un objeto anidado y `etiquetas` es un arreglo:

```json
{
    "producto": "Laptop",
    "precio": 850,
    "disponible": true,
    "categoria": {
        "id": 1,
        "nombre": "Tecnologia"
    },
    "etiquetas": ["computacion", "electronica"]
}
```

Una tabla tiene filas y columnas de un solo nivel; un JSON puede tener **varios niveles**. Por eso no siempre se puede convertir directamente en un DataFrame.

Se observa el inicio del archivo `productos.json`:

In [ ]:
print((DATA_DIR / "productos.json").read_text(encoding="utf-8")[:700])

### 4.2. Lectura de JSON con Pandas

Cuando el archivo es una **lista de objetos**, `pd.read_json()` construye un DataFrame donde cada objeto es una fila:

In [ ]:
df_json = pd.read_json(
    DATA_DIR / "productos.json"
)
df_json.head()

Las columnas `categoria` y `caracteristicas` contienen **diccionarios completos** dentro de cada celda, y `etiquetas` contiene listas. El DataFrame se creó, pero no es una tabla útil para analizar:

In [ ]:
print(type(df_json.loc[0, "categoria"]))
df_json.loc[0, "categoria"]

### 4.3. Lectura e interpretación con Python

El módulo `json` carga el archivo respetando su estructura original. Esto permite **explorar** la estructura antes de decidir cómo convertirla en tabla.

In [ ]:
with open(DATA_DIR / "productos.json", "r", encoding="utf-8") as file:
    data = json.load(file)

type(data)

In [ ]:
print("Cantidad de elementos:", len(data))
print("Tipo del primer elemento:", type(data[0]))
print("Claves del primer elemento:", list(data[0].keys()))

In [ ]:
# Acceso a valores anidados: se encadenan las claves
primer_producto = data[0]
print("Producto  :", primer_producto["producto"])
print("Categoría :", primer_producto["categoria"]["nombre"])
print("Ancho (cm):", primer_producto["caracteristicas"]["dimensiones"]["ancho_cm"])
print("Etiquetas :", primer_producto["etiquetas"])

### 4.4. Normalización de estructuras anidadas

`pd.json_normalize()` "aplana" los objetos anidados: cada clave interna se convierte en una columna, uniendo los niveles con un punto (`categoria.nombre`).

In [ ]:
df_normalizado = pd.json_normalize(data)
df_normalizado.head()

In [ ]:
df_normalizado.columns.tolist()

Tres observaciones importantes:

1. Los **objetos** anidados (`categoria`, `caracteristicas`, `dimensiones`) se convirtieron en columnas.
2. Los **arreglos** (`etiquetas`) **no** se aplanan: siguen como listas dentro de la celda.
3. Si una clave no existe en todos los objetos (`peso_kg`), aparece como `NaN` en los registros donde falta.

In [ ]:
df_normalizado["caracteristicas.peso_kg"].isnull().sum()

#### No todos los JSON pueden convertirse directamente en una tabla

El archivo `catalogo.json` tiene una estructura distinta: los productos están dentro de una clave, acompañados de metadatos.

In [ ]:
with open(DATA_DIR / "catalogo.json", "r", encoding="utf-8") as file:
    catalogo = json.load(file)

print("Tipo:", type(catalogo))
print("Claves de primer nivel:", list(catalogo.keys()))

Si se intenta leer directamente con Pandas, el resultado es un error o una tabla sin sentido, porque Pandas no sabe cuál de las claves contiene los registros:

In [ ]:
try:
    df_catalogo = pd.read_json(DATA_DIR / "catalogo.json")
    display(df_catalogo)
except Exception as error:
    print("Error durante la lectura:", type(error).__name__)
    print(error)

Primero hay que **interpretar** la estructura e indicar dónde están los registros. Además, cada producto tiene una lista de proveedores, que también es una estructura anidada:

In [ ]:
# Opción 1: seleccionar la lista de registros
pd.json_normalize(catalogo["productos"])

In [ ]:
# Opción 2: una fila por proveedor, conservando datos del producto y del catálogo
df_proveedores = pd.json_normalize(
    catalogo["productos"],
    record_path="proveedores",                         # lista que se expande en filas
    meta=["id_producto", "producto"],                  # datos del nivel superior
    record_prefix="proveedor."
)
df_proveedores["empresa"] = catalogo["empresa"]
df_proveedores

> **Idea clave:** antes de convertir un JSON en DataFrame hay que responder tres preguntas: ¿el primer nivel es un objeto o una lista?, ¿dónde están los registros?, ¿qué campos están anidados y cómo se representarán?

---
## 5. Extracción desde una API REST

> **Sección central de la práctica.** Requiere conexión a internet.

**Objetivo:** obtener datos desde un servicio externo mediante solicitudes HTTP.

### 5.1. Conceptos fundamentales

- **API** (*Application Programming Interface*): conjunto de reglas que permite que un programa solicite datos o servicios a otro programa sin conocer su funcionamiento interno.
- **API REST:** estilo de diseño de APIs que usa el protocolo HTTP. Cada recurso (productos, usuarios, publicaciones) se identifica con una URL y se manipula con métodos HTTP estándar. Normalmente responde en formato JSON.
- **Cliente y servidor:** el **cliente** realiza la solicitud (nuestro código en Python); el **servidor** la procesa y devuelve una respuesta.
- **Endpoint:** URL específica que da acceso a un recurso. Ejemplo: `https://jsonplaceholder.typicode.com/posts`.
- **Solicitud y respuesta:** la solicitud contiene el método, la URL, parámetros y encabezados; la respuesta contiene un **código de estado**, encabezados y un **cuerpo** con los datos.

**Métodos HTTP**

| Método | Acción | Ejemplo |
|---|---|---|
| `GET` | Obtener datos | Consultar la lista de productos |
| `POST` | Crear un recurso | Registrar un producto nuevo |
| `PUT` | Actualizar un recurso | Modificar el precio de un producto |
| `DELETE` | Eliminar un recurso | Dar de baja un producto |

En extracción de datos se usa principalmente **GET**, porque el objetivo es **leer** información sin modificarla.

**Códigos de estado HTTP:** número de tres dígitos que indica el resultado de la solicitud. Se agrupan por su primer dígito: `2xx` éxito, `4xx` error del cliente, `5xx` error del servidor. Se detallan en la sección 5.4.

### 5.2. Estructura de una solicitud

```
 ┌───────────────────────┐                            ┌───────────────────────┐
 │   Python / Cliente    │  1. requests.get(url)      │   API REST / Servidor │
 │                       │ ─────────────────────────> │                       │
 │                       │                            │ 2. Procesamiento de   │
 │                       │                            │    la solicitud       │
 │                       │  3. Respuesta JSON         │                       │
 │ 4. Datos y código de  │ <───────────────────────── │                       │
 │    estado             │                            │                       │
 └───────────────────────┘                            └───────────────────────┘
```

1. El cliente envía una solicitud `GET` al endpoint.
2. El servidor procesa la solicitud (valida, consulta su base de datos).
3. El servidor devuelve una respuesta con el cuerpo en formato JSON.
4. El cliente recibe el código de estado y los datos, y los convierte en una estructura de Python.

### 5.3. Primera extracción mediante GET

Se utiliza **JSONPlaceholder**, una API pública de prueba que no requiere registro ni credenciales.

Endpoint: `https://jsonplaceholder.typicode.com/posts`

El parámetro `timeout=10` indica que, si el servidor no responde en 10 segundos, la solicitud se cancela. Sin este parámetro, el programa podría quedar esperando indefinidamente.

In [ ]:
url = "https://jsonplaceholder.typicode.com/posts"
response = requests.get(url, timeout=10)
response.status_code

### 5.4. Interpretación de la respuesta

El objeto `response` contiene toda la información devuelta por el servidor:

In [ ]:
response.status_code

In [ ]:
# Encabezados: metadatos de la respuesta (tipo de contenido, fecha, servidor...)
response.headers

In [ ]:
print("Tipo de contenido:", response.headers.get("Content-Type"))

In [ ]:
# Cuerpo de la respuesta como texto (primeros 500 caracteres)
response.text[:500]

**Códigos de estado principales**

| Código | Significado |
|---|---|
| 200 | Solicitud exitosa |
| 201 | Recurso creado |
| 400 | Solicitud incorrecta |
| 401 | No autorizado |
| 403 | Acceso prohibido |
| 404 | Recurso no encontrado |
| 500 | Error interno del servidor |

Ejemplo de un recurso que no existe:

In [ ]:
respuesta_inexistente = requests.get(f"{url}/99999", timeout=10)
print("Código de estado:", respuesta_inexistente.status_code)
print("¿Solicitud exitosa?:", respuesta_inexistente.ok)

### 5.5. Conversión de la respuesta JSON

`response.text` es una cadena de texto. El método `response.json()` la interpreta y la convierte en estructuras de Python (listas y diccionarios), igual que `json.load()` con un archivo.

In [ ]:
data_api = response.json()
type(data_api)

In [ ]:
print("Cantidad de registros:", len(data_api))
data_api[0]

Como la respuesta es una lista de objetos sin anidamiento, puede convertirse directamente en DataFrame:

In [ ]:
df_api = pd.DataFrame(data_api)
df_api.head()

### 5.6. Extracción selectiva mediante parámetros

Los **parámetros de consulta** (*query parameters*) permiten pedir al servidor solo los datos necesarios. `requests` los agrega a la URL automáticamente.

In [ ]:
params = {
    "userId": 1
}

response = requests.get(
    url,
    params=params,
    timeout=10
)

print("URL consultada:", response.url)

data_api = response.json()
df_api = pd.DataFrame(data_api)
print("Registros obtenidos:", len(df_api))
df_api.head()

Filtrar en el servidor es más eficiente que descargar todo y filtrar después: se transfieren menos datos y el procesamiento ocurre donde están los datos.

### 5.7. Manejo básico de errores

Una extracción desde una API puede fallar por muchas causas: el servidor no responde, no hay internet, el recurso no existe. El bloque `try / except` permite **controlar** el error en lugar de que el programa se detenga.

- `timeout=10`: limita el tiempo de espera.
- `raise_for_status()`: genera una excepción si el código de estado es `4xx` o `5xx`.
- `requests.exceptions.RequestException`: clase base que agrupa todos los errores de `requests` (conexión, tiempo de espera, HTTP).

In [ ]:
try:
    response = requests.get(
        url,
        timeout=10
    )
    response.raise_for_status()
    data_api = response.json()
    df_api = pd.DataFrame(data_api)
    print("Extracción exitosa:", df_api.shape)
except requests.exceptions.RequestException as error:
    print("Error durante la extracción:", error)

El mismo bloque con un endpoint inexistente muestra cómo se captura el error sin detener el notebook:

In [ ]:
try:
    response_error = requests.get(
        "https://jsonplaceholder.typicode.com/productos-inexistentes",
        timeout=10
    )
    response_error.raise_for_status()
    datos = response_error.json()
except requests.exceptions.RequestException as error:
    print("Error durante la extracción:", error)

### 5.8. Limitaciones de la extracción mediante API

| Limitación | Descripción |
|---|---|
| Disponibilidad del servicio | Si el servidor está caído o en mantenimiento, no hay datos. |
| Límites de solicitudes | Muchas APIs limitan cuántas solicitudes se pueden hacer por minuto u hora (*rate limiting*); al superarlo responden con código `429`. |
| Autenticación | Las APIs privadas exigen una clave (*API key*) o un token; sin ellos se obtiene `401` o `403`. |
| Paginación | Los resultados se entregan por partes (páginas); para obtener todo hay que hacer varias solicitudes. |
| Cambios en la estructura de respuesta | El proveedor puede renombrar o eliminar campos, lo que rompe el código que los utiliza. |
| Conectividad | Sin acceso a internet, o con una red lenta o restringida, la extracción falla. |

**Ejemplo de paginación.** JSONPlaceholder acepta los parámetros `_page` y `_limit`. El encabezado `x-total-count` indica cuántos registros existen en total:

In [ ]:
respuesta_pagina = requests.get(url, params={"_page": 1, "_limit": 10}, timeout=10)

print("Registros en esta página:", len(respuesta_pagina.json()))
print("Registros totales disponibles:", respuesta_pagina.headers.get("x-total-count"))

In [ ]:
# Recorrido de todas las páginas hasta que el servidor devuelva una lista vacía
registros = []
pagina = 1

while True:
    r = requests.get(url, params={"_page": pagina, "_limit": 25}, timeout=10)
    r.raise_for_status()
    lote = r.json()
    if not lote:
        break
    registros.extend(lote)
    print(f"Página {pagina}: {len(lote)} registros")
    pagina += 1

print("Total extraído:", len(registros))

---
## 6. Comparación de fuentes de datos

**Objetivo:** identificar las diferencias entre los mecanismos de extracción utilizados.

| Característica | CSV | Excel | JSON | API REST |
|---|---|---|---|---|
| Fuente | Archivo | Archivo | Archivo | Servicio externo |
| Estructura | Tabular | Hojas y tablas | Jerárquica | Depende de respuesta |
| Actualización | Manual | Manual | Manual | Puede ser dinámica |
| Conectividad | No requiere | No requiere | No requiere | Generalmente requiere |
| Función principal | `pd.read_csv()` | `pd.read_excel()` | `json.load()` / `pd.json_normalize()` | `requests.get()` |

### Actividad de interpretación

1. ¿Qué fuente utilizaría una empresa para obtener sus ventas históricas?
2. ¿Qué fuente sería conveniente para consultar información actualizada de un servicio externo?
3. ¿Qué formato permite representar estructuras anidadas?
4. ¿Por qué no es correcto asumir que toda respuesta JSON tiene una estructura tabular?

**Respuesta:**

_Escriba aquí su respuesta._

---
## 7. Validación de los datos extraídos

**Objetivo:** comprobar que la extracción produjo un conjunto de datos utilizable.

Esta sección conecta directamente con los notebooks anteriores: se aplican las mismas técnicas de inspección, ahora como **control de calidad de la extracción**. Se trabaja con `df_api`, el DataFrame obtenido desde JSONPlaceholder.

### 7.1. Inspección inicial

In [ ]:
df_api.head()

In [ ]:
df_api.shape

In [ ]:
df_api.columns

In [ ]:
df_api.dtypes

In [ ]:
df_api.info()

### 7.2. Verificación de valores nulos

In [ ]:
df_api.isnull().sum()

### 7.3. Verificación de duplicados

`duplicated()` detecta filas completamente repetidas. También conviene verificar que el **identificador** sea único:

In [ ]:
print("Filas duplicadas         :", df_api.duplicated().sum())
print("Identificadores repetidos:", df_api["id"].duplicated().sum())

### 7.4. Verificación de registros y estructura

Elementos a verificar:

- Cantidad de registros obtenidos.
- Columnas disponibles.
- Tipos de datos.
- Campos faltantes.
- Registros repetidos.
- Diferencias entre la estructura esperada y la obtenida.

Se define una **estructura esperada** (según la documentación de la API) y se compara con la obtenida:

In [ ]:
estructura_esperada = {
    "userId": "int64",
    "id": "int64",
    "title": "object",
    "body": "object",
}
registros_esperados = 100

columnas_obtenidas = set(df_api.columns)
columnas_esperadas = set(estructura_esperada)

print("Registros esperados / obtenidos:", registros_esperados, "/", len(df_api))
print("Columnas faltantes :", columnas_esperadas - columnas_obtenidas or "ninguna")
print("Columnas adicionales:", columnas_obtenidas - columnas_esperadas or "ninguna")

for columna, tipo in estructura_esperada.items():
    if columna in df_api.columns:
        tipo_real = str(df_api[columna].dtype)
        if tipo == "object":
            estado = "OK" if tipo_real in ("object", "str", "string") else "REVISAR"
        else:
            estado = "OK" if tipo_real == tipo else "REVISAR"
        print(f"  {columna:<8} esperado={tipo:<7} obtenido={tipo_real:<7} {estado}")

Para reutilizar estas verificaciones con cualquier fuente, se agrupan en una función. Se usará en la práctica integradora.

In [ ]:
def resumen_extraccion(df: pd.DataFrame, nombre: str) -> pd.DataFrame:
    """Devuelve un resumen de calidad inicial de un DataFrame extraído."""
    print(f"Fuente: {nombre}")
    print(f"  Registros            : {df.shape[0]}")
    print(f"  Columnas             : {df.shape[1]}")
    print(f"  Filas duplicadas     : {df.astype(str).duplicated().sum()}")
    print(f"  Celdas con nulos     : {int(df.isnull().sum().sum())}")
    return pd.DataFrame({
        "tipo": df.dtypes.astype(str),
        "nulos": df.isnull().sum(),
        "valores_unicos": df.astype(str).nunique(),
    })


resumen_extraccion(df_api, "API JSONPlaceholder - posts")

> Nota: en `resumen_extraccion` se convierte temporalmente el DataFrame a texto (`astype(str)`) para contar duplicados y valores únicos. Esto evita errores cuando alguna columna contiene listas o diccionarios, que es frecuente en datos provenientes de JSON.

### 7.5. Reflexión

¿Por qué una extracción exitosa técnicamente (código `200`, sin errores) no garantiza que los datos sean adecuados para un análisis?

**Respuesta:**

_Escriba aquí su respuesta._

---
## 8. Práctica integradora: extracción desde múltiples fuentes

**Objetivo:** desarrollar un pequeño flujo de extracción documentado.

### Escenario

Una empresa necesita recopilar información de productos para elaborar un análisis de inventario. Los datos están distribuidos en tres fuentes:

| N.º | Fuente | Contenido | Ubicación |
|---|---|---|---|
| 1 | Archivo CSV | Productos y precios | `productos.csv` |
| 2 | Archivo JSON | Categorías y características | `productos.json` |
| 3 | API REST | Información complementaria de productos | `https://dummyjson.com/products` |

La API **DummyJSON** es un servicio público de prueba que no requiere credenciales y devuelve un catálogo de productos.

### Actividades

1. Leer los archivos CSV y JSON.
2. Consultar el endpoint proporcionado.
3. Convertir las respuestas a estructuras tabulares cuando corresponda.
4. Inspeccionar los conjuntos obtenidos.
5. Identificar diferencias entre las estructuras.
6. Documentar los problemas encontrados durante la extracción.

**Producto esperado:** notebook con el proceso de extracción documentado, resultados de cada fuente y una breve evaluación de la calidad inicial de los datos.

Durante el flujo se usa una lista llamada `registro_problemas` para anotar, de forma automática, las situaciones detectadas en cada fuente.

In [ ]:
registro_problemas = []


def registrar(fuente: str, problema: str, evidencia: str) -> None:
    registro_problemas.append({"fuente": fuente, "problema": problema, "evidencia": evidencia})

### 8.1. Actividad 1 — Lectura del archivo CSV

In [ ]:
fuente = "CSV - productos.csv"

df_inv_csv = pd.read_csv(DATA_DIR / "productos.csv", encoding="utf-8")
df_inv_csv.head()

### 8.2. Actividad 1 — Lectura del archivo JSON

Primero se interpreta la estructura y luego se normaliza.

In [ ]:
with open(DATA_DIR / "productos.json", "r", encoding="utf-8") as file:
    data_inv_json = json.load(file)

print("Tipo de primer nivel:", type(data_inv_json).__name__)
print("Cantidad de elementos:", len(data_inv_json))
print("Claves de un elemento:", list(data_inv_json[0].keys()))

In [ ]:
df_inv_json = pd.json_normalize(data_inv_json)
df_inv_json.head()

### 8.3. Actividades 2 y 3 — Consulta del endpoint y conversión a tabla

Se solicitan 30 productos y solo los campos necesarios mediante los parámetros `limit` y `select`.

In [ ]:
url_productos = "https://dummyjson.com/products"
params_productos = {
    "limit": 30,
    "select": "title,category,brand,price,stock,rating,dimensions",
}

df_inv_api = pd.DataFrame()
respuesta_api = None

try:
    respuesta_api = requests.get(url_productos, params=params_productos, timeout=10)
    respuesta_api.raise_for_status()
    data_inv_api = respuesta_api.json()
    print("Código de estado:", respuesta_api.status_code)
    print("Tipo de primer nivel:", type(data_inv_api).__name__)
    print("Claves de primer nivel:", list(data_inv_api.keys()))
except requests.exceptions.RequestException as error:
    data_inv_api = None
    print("Error durante la extracción:", error)
    registrar("API - DummyJSON", "La consulta al endpoint falló", str(error))

La respuesta **no es una lista** sino un objeto que "envuelve" los registros junto con información de paginación (`total`, `skip`, `limit`). Igual que con `catalogo.json`, hay que indicar dónde están los registros antes de construir la tabla.

In [ ]:
if data_inv_api is not None:
    print("Total de productos en la API:", data_inv_api.get("total"))
    print("Productos recibidos          :", len(data_inv_api.get("products", [])))
    print("Desplazamiento (skip)        :", data_inv_api.get("skip"))

    if data_inv_api.get("total", 0) > len(data_inv_api.get("products", [])):
        registrar("API - DummyJSON",
                  "Respuesta paginada: no se obtuvo el catálogo completo",
                  f"{len(data_inv_api['products'])} de {data_inv_api['total']} productos")

    df_inv_api = pd.json_normalize(data_inv_api["products"])

df_inv_api.head()

### 8.4. Actividad 4 — Inspección de los conjuntos obtenidos

In [ ]:
resumen_extraccion(df_inv_csv, "CSV - productos.csv")

In [ ]:
resumen_extraccion(df_inv_json, "JSON - productos.json")

In [ ]:
if not df_inv_api.empty:
    display(resumen_extraccion(df_inv_api, "API - DummyJSON"))
else:
    print("No se obtuvieron datos de la API.")

Se registran automáticamente los problemas de calidad detectados en cada fuente:

In [ ]:
fuentes = {
    "CSV - productos.csv": df_inv_csv,
    "JSON - productos.json": df_inv_json,
    "API - DummyJSON": df_inv_api,
}

for nombre, df in fuentes.items():
    if df.empty:
        continue
    duplicados = df.astype(str).duplicated().sum()
    if duplicados:
        registrar(nombre, "Registros duplicados", f"{duplicados} filas repetidas")
    for columna, cantidad in df.isnull().sum().items():
        if cantidad:
            registrar(nombre, f"Valores nulos en '{columna}'", f"{cantidad} registros")
    for columna in df.columns:
        if df[columna].apply(lambda v: isinstance(v, (list, dict))).any():
            registrar(nombre, f"La columna '{columna}' contiene listas o diccionarios",
                      "No es un valor atómico; requerirá tratamiento posterior")

print("Problemas registrados:", len(registro_problemas))

### 8.5. Actividad 5 — Diferencias entre las estructuras

Se construye una tabla que muestra qué columnas tiene cada fuente:

In [ ]:
todas_columnas = sorted(set().union(*[df.columns for df in fuentes.values()]))

comparacion = pd.DataFrame(
    {nombre: ["X" if c in df.columns else "" for c in todas_columnas]
     for nombre, df in fuentes.items()},
    index=todas_columnas,
)
comparacion

In [ ]:
# Nombre que recibe un mismo concepto en cada fuente
equivalencias = pd.DataFrame({
    "Concepto": ["Identificador", "Nombre del producto", "Precio", "Stock", "Categoría"],
    "CSV": ["id_producto", "producto", "precio", "stock", "categoria_id"],
    "JSON": ["id_producto", "producto", "-", "-", "categoria.nombre"],
    "API": ["id", "title", "price", "stock", "category"],
})
equivalencias

Algunas diferencias que deben analizarse:

- **Nombres de columnas:** el mismo concepto tiene nombres distintos (`producto` / `title`, `precio` / `price`).
- **Idioma:** las fuentes locales están en español; la API en inglés, incluso en los valores (`category`).
- **Representación de la categoría:** el CSV usa un código numérico (`categoria_id`), el JSON usa un objeto con código y nombre, y la API un texto.
- **Granularidad:** el JSON incluye características anidadas (dimensiones, garantía) que no existen en el CSV.
- **Cobertura:** el CSV tiene 122 filas, el JSON 30 y la API entrega una página de su catálogo; además, los productos de la API **no corresponden** a los de la empresa.

### 8.6. Actividad 6 — Documentación de problemas encontrados

Registro generado durante el flujo:

In [ ]:
df_problemas = pd.DataFrame(registro_problemas)
df_problemas

Complete la siguiente tabla con su análisis. Puede incluir problemas del registro anterior y otros que haya identificado (por ejemplo, diferencias de nombres o de idioma).

| Fuente | Problema encontrado | Evidencia | Posible impacto en el análisis |
|---|---|---|---|
| | | | |
| | | | |
| | | | |
| | | | |

### 8.7. Evaluación de la calidad inicial de los datos

Redacte una breve evaluación (5 a 8 líneas) que responda:

- ¿Qué fuente presenta mejor calidad inicial y por qué?
- ¿Qué fuente requerirá más trabajo de preparación?
- ¿Es posible combinar las tres fuentes directamente? ¿Qué se necesitaría?

**Respuesta:**

_Escriba aquí su respuesta._

---
## 9. Conclusiones y evaluación

### Síntesis

- La **extracción** es la primera etapa del proceso ETL y consiste en obtener los datos desde su fuente y cargarlos correctamente en el entorno de trabajo.
- Los archivos **CSV** son simples y universales, pero exigen conocer el separador, la codificación y la ubicación de los encabezados.
- Los archivos **Excel** conservan tipos de datos y organizan la información en hojas, pero pueden contener títulos, celdas vacías y varias tablas que obligan a extraer de forma selectiva.
- Los archivos **JSON** representan estructuras jerárquicas; antes de convertirlos en tabla hay que interpretar su estructura y normalizarla.
- Las **API REST** permiten obtener datos actualizados desde servicios externos, pero dependen de la conectividad, la disponibilidad del servicio, la autenticación y la paginación.
- Una extracción técnicamente exitosa **no garantiza** datos de calidad: siempre debe verificarse la estructura, los tipos, los nulos y los duplicados.

Los datos no necesariamente llegan en un CSV listo para Pandas: pueden estar distribuidos en archivos, estructuras jerárquicas o servicios externos.

### Preguntas finales

1. ¿Qué diferencia existe entre leer un archivo y consumir una API?
2. ¿Por qué es importante verificar el código de estado HTTP?
3. ¿Qué ventajas ofrece JSON frente a CSV para representar estructuras complejas?
4. ¿Qué problemas pueden presentarse durante una extracción?
5. ¿Por qué la extracción constituye una etapa fundamental antes del análisis exploratorio?

**Respuesta:**

_Escriba aquí su respuesta._